# 预构建中间件

LangChain 和[Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview)为常见用例提供预构建的中间件。每个中间件都已准备好投入生产环境，并且可以根据您的特定需求进行配置。

## 与提供商无关的中间件

预构建中间件是 LangChain 提供的“通用拦截层”，但是否生效取决于模型能力。



### Summarization 总结/摘要中间件

当接近消息数量上限时，自动生成对话历史记录摘要，保留最近的消息，同时压缩较早的上下文。摘要功能适用于以下情况：

- 持续时间过长的对话，超出上下文窗口。
- 多回合对话，历史悠久。
- 需要保留完整对话上下文的应用场景。

 `SummarizationMiddleware` = **自动帮你压缩上下文，防止 token 爆炸**，本质是在做**长上下文的自动内存管理**。

示例



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model

summarization_model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama"
)
model = init_chat_model(model="gemma4:e4b", model_provider="ollama")

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=summarization_model,
            trigger=[("tokens", 4000), ("messages", 6)],
            keep=("messages", 20),
        ),
    ],
)


#### 参数解析

| 参数    | 本质                 |
| ------- | -------------------- |
| model   | 谁来做“压缩”         |
| trigger | 什么时候压           |
| keep    | 压完保留多少原始细节 |

##### 1️⃣ model

```
model=summarization_model
```

👉 **用来“做总结”的模型**

不是主模型！

```
summarization_model = qwen2.5:7b
model = gemma4:e4b
```

👉 含义是：

- 主 Agent：用 `gemma`
- 总结：用 `qwen`

✔️ 好处：

- 省成本（总结可以用便宜模型）
- 解耦（总结逻辑独立）

------

##### 2️⃣ trigger

```
trigger=("tokens", 4000)
```

👉 **什么时候触发“总结”**

格式是：

```
(type, threshold)
```

```
("tokens", 4000)
```

👉 意思：

```
当上下文 token 数 > 4000 → 触发总结
```

------

##### 🔁 messages：

```
("messages", 50)  # 超过50条消息触发
```

------

##### 3️⃣ keep

```
keep=("messages", 20)
```

👉 **总结后，保留多少“原始上下文”**

```
总结完成后：
- 保留最近 20 条消息（不压缩）
- 更早的 → 被总结成一段 summary
```

------

#### 🧠 实际效果：

原始：

```
msg1, msg2, ..., msg100
```

变成：

```
summary(msg1~msg80) + msg81~msg100
```

### Human-in-loop 人机交互

在工具调用执行前，暂停代理程序的执行，以便人工审批、编辑或拒绝这些调用。[人机协作机制](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)在以下情况下非常有用：

- 需要人工批准的高风险操作（例如数据库写入、金融交易）。
- 需要人工监督的合规工作流程。
- 长时间的对话，其中人类的反馈会指导智能体。

#### 参数解析

##### interrupt_on

哪些工具调用需要被“拦截”

###### 写法一：决策范围

| 决策    | 含义                       |
| ------- | -------------------------- |
| approve | ✅ 同意执行（原样调用工具） |
| edit    | ✏️ 修改参数再执行           |
| reject  | ❌ 拒绝调用                 |

👉 含义：

当 Agent 想调用 `send_email` 时：

```
暂停执行 → 等人类决定
```

------

###### 🔘 allowed_decisions

```
"allowed_decisions": ["approve", "edit", "reject"]
```

👉 定义：**人类可以做哪些操作**

###### 写法2：不拦截（False）

```
"your_read_email_tool": False
```

👉 含义：

```
这个工具直接放行，不需要人工干预
```

##### 状态保存：checkpointer

```
checkpointer=InMemorySaver()
```

👉 这个是 **必须配合 HumanInTheLoop 才有意义的**， 把 Agent 当前执行状态存下来（内存版）

* 为什么

因为流程会被“暂停”：

```
模型 → 准备调工具 → ❗被中断 → 等人 → 再继续
```

👉 所以需要：

```
保存当前状态（checkpoint）
```



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def your_read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def your_send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[your_read_email_tool, your_send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "your_send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "your_read_email_tool": False,
            }
        ),
    ],
)

### Model call limit 模型调用限制

限制模型调用次数以防止无限循环或过高的成本。模型调用次数限制在以下情况下非常有用：

- 防止失控代理发出过多 API 调用。
- 对生产部署实施成本控制。
- 在特定通话预算范围内测试客服人员的行为。

#### 参数解析

| 参数名称 | 类型          | 含义                                                   | 默认值  | 触发时行为                             |
| -------- | ------------- | ------------------------------------------------------ | ------- | -------------------------------------- |
| 线程限制 | 数字（int）   | 一个线程（thread）内**所有运行累计的模型调用次数上限** | 无限制  | 超过后根据“退出行为”处理               |
| 运行限制 | 数字（int）   | 单次 Agent 调用中**模型调用次数的上限**                | 无限制  | 超过后根据“退出行为”处理               |
| 退出行为 | 字符串（str） | 达到限制时的处理方式                                   | `"end"` | `"end"`：正常终止；`"error"`：抛出异常 |



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=10,
            run_limit=5,
            exit_behavior="end",
        ),
    ],
)

###  Tool call limit 工具调用限制

通过限制工具调用次数来控制代理的执行，可以全局限制所有工具，也可以限制特定工具。工具调用次数限制在以下情况下非常有用：

- 防止过度调用昂贵的外部 API。
- 限制网络搜索或数据库查询。
- 对特定工具的使用频率实施限制。
- 防止代理程序陷入失控循环。

####  参数解析

| 参数名称                 | 类型               | 含义                                                         | 默认值         | 作用范围                | 触发后的行为           |
| ------------------------ | ------------------ | ------------------------------------------------------------ | -------------- | ----------------------- | ---------------------- |
| 工具名称                 | 字符串（str）      | 指定要限制的工具名称；不填则作用于所有工具                   | None           | 单个工具 / 全部工具     | 决定限制作用对象       |
| 线程限制（thread_limit） | 数字（int / None） | 一个线程（会话）内**累计的工具调用总次数上限**（跨多次调用） | None（无限制） | 跨请求（需 checkpoint） | 超过后按“退出行为”处理 |
| 运行限制（run_limit）    | 数字（int / None） | 单次调用（一次用户请求）内**工具调用次数上限**               | None（无限制） | 单次请求                | 每次新用户消息会重置   |
| 退出行为（behavior）     | 字符串（str）      | 达到限制时的处理方式                                         | `"continue"`   | —                       | 见下方详细说明         |

------

##### ⚙️ 退出行为（behavior）详细说明

| 值                   | 行为     | 说明                                                         |
| -------------------- | -------- | ------------------------------------------------------------ |
| `"continue"`（默认） | 软限制   | 阻止超限工具调用，返回错误信息给模型，**模型可以继续运行并决定结束** |
| `"error"`            | 强制中断 | 抛出 `ToolCallLimitExceededError`，**立即停止执行**          |
| `"end"`              | 优雅结束 | 立即终止执行，并返回提示信息（仅适用于**单工具限制场景**，否则可能抛异常） |

------

* 注意

```
thread_limit 和 run_limit 至少要设置一个
```

否则这个限制配置是无效的。



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.tools import tool

@tool
def search_tool():
    """Support search"""
    pass

@tool
def database_tool():
    """Support database CURD"""
    pass

agent = create_agent(
    model="gpt-4.1",
    tools=[search_tool, database_tool],
    middleware=[
        # Global limit
        ToolCallLimitMiddleware(thread_limit=20, run_limit=10),
        # Tool-specific limit
        ToolCallLimitMiddleware(
            tool_name="search",
            thread_limit=5,
            run_limit=3,
        ),
    ],
)

### Model fallback 模型回退

当主模型失效时，自动回退到备用模型。模型回退功能适用于以下情况：

- 构建能够处理模型故障的弹性代理。
- 通过采用更便宜的型号来优化成本。



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        ModelFallbackMiddleware(
            summarization_model
        ),
    ],
)

### PII 检测

使用可配置策略检测和处理对话中的个人身份信息 (PII)。PII 检测可用于以下用途：

- 医疗保健和金融应用，需满足合规性要求。
- 需要清理日志的客服人员。
- 任何处理敏感用户数据的应用程序。

#### 参数解析

| 参数名称              | 类型              | 含义                             | 默认值     | 可选值 / 说明                                                |
| --------------------- | ----------------- | -------------------------------- | ---------- | ------------------------------------------------------------ |
| pii_type              | 字符串（str）     | 要检测的 PII（个人敏感信息）类型 | 必填       | 内置类型（如 `email`、`credit_card`、`ip`、`mac_address`、`url`），或自定义类型 |
| strategy              | 字符串（str）     | 检测到 PII 后的处理方式          | `"redact"` | `'block'`（抛异常）、`'redact'`（替换为 `[REDACTED_xxx]`）、`'mask'`（部分遮盖）、`'hash'`（哈希替换） |
| detector              | 函数 / 正则表达式 | 自定义检测逻辑                   | None       | 不提供则使用内置检测器                                       |
| apply_to_input        | 布尔（bool）      | 是否在模型调用前检查用户输入     | True       | 控制输入侧安全                                               |
| apply_to_output       | 布尔（bool）      | 是否在模型输出后检查 AI 回复     | False      | 控制输出泄露                                                 |
| apply_to_tool_results | 布尔（bool）      | 是否在工具执行后检查结果         | False      | 控制外部数据源返回                                           |

 **pii_type 定义“查什么”，strategy 定义“怎么处理”，apply_\* 定义“在哪查”**

| 阶段        | 参数                  |
| ----------- | --------------------- |
| 用户 → 模型 | apply_to_input        |
| 模型 → 用户 | apply_to_output       |
| 工具 → 模型 | apply_to_tool_results |



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
)

### LLM工具选择器

使用 LLM 在调用主模型之前智能地选择相关工具。LLM 工具选择器可用于以下用途：

- 代理拥有许多工具（10 个以上），但大多数工具与每次查询都不相关。
- 通过过滤不相关的工具来减少令牌使用量。
- 提高模型聚焦性和准确性。

该中间件使用结构化输出向 LLM 询问哪些工具与当前查询最相关。结构化输出模式定义了可用工具的名称和描述。模型提供者通常会在后台将这些结构化输出信息添加到系统提示中。



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware

agent = create_agent(
    model="gpt-4.1",
    tools=[],#tool1, tool2, tool3, tool4, tool5, ...]
    middleware=[
        LLMToolSelectorMiddleware(
            model="gpt-4.1-mini",
            max_tools=3,
            always_include=["search"],
        ),
    ],
)

### Tools Retry 工具重试

使用可配置的指数退避算法自动重试失败的工具调用。工具重试功能适用于以下情况：

- 处理外部 API 调用中的瞬态故障。
- 提高网络依赖型工具的可靠性。
- 构建能够优雅地处理临时错误的弹性代理。

#### 参数解析

| 参数名称                   | 类型                  | 含义                             | 默认值                 | 说明                          |
| -------------------------- | --------------------- | -------------------------------- | ---------------------- | ----------------------------- |
| 最大重试次数               | 数字（int）           | 工具首次调用失败后，最多重试次数 | 2（通常总尝试=1+重试） | 控制重试上限                  |
| 工具（tools）              | list[BaseTool \| str] | 指定要应用重试逻辑的工具         | None                   | None 表示对所有工具生效       |
| 重试（retry_on）           | 异常类型元组 / 可调用 | 哪些异常触发重试                 | `(Exception,)`         | 也可传函数：`fn(exc) -> bool` |
| 失败（on_failure）         | 字符串 / 可调用       | 所有重试失败后的处理方式         | `"return_message"`     | 见下方详细说明                |
| 回退因子（backoff_factor） | 数字（float）         | 指数退避倍数                     | 2.0                    | 控制重试间隔增长速度          |
| 初始延迟（initial_delay）  | 数字（float）         | 第一次重试前等待时间（秒）       | 1.0                    | 基础延迟                      |
| 最大延迟（max_delay）      | 数字（float）         | 重试间隔最大值（秒）             | 60.0                   | 防止等待时间过长              |
| 抖动（jitter）             | 布尔（bool）          | 是否增加随机波动（±25%）         | True                   | 避免“惊群效应”                |

##### 失败处理（on_failure）详解

| 值                         | 行为         | 说明                                            |
| -------------------------- | ------------ | ----------------------------------------------- |
| `"return_message"`（默认） | 返回错误信息 | 生成一个 `ToolMessage` 给 LLM，让模型决定下一步 |
| `"raise"`                  | 抛异常       | 直接中断 Agent 执行                             |
| 自定义函数                 | 自定义处理   | `fn(exception) -> str`，返回错误消息内容        |

##### 指数退避公式

👉 每次重试等待时间：

```
delay = initial_delay * (backoff_factor ^ retry_number)
```

并且：

```
最终 delay ≤ max_delay
```

如果开启 jitter：

```
实际 delay = delay ± 25% 随机波动
```



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware

agent = create_agent(
    model="gpt-4.1",
    tools=[search_tool, database_tool],
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,
            backoff_factor=2.0,
            initial_delay=1.0,
        ),
    ],
)

### Models Retry模型重试

使用可配置的指数退避算法自动重试失败的模型调用。模型重试功能适用于以下情况：

- 处理模型 API 调用中的瞬态故障。
- 提高网络相关模型请求的可靠性。
- 构建能够优雅地处理临时模型错误的弹性代理。

#### 参数解析

| 参数名称                   | 类型                  | 含义                             | 默认值                 | 说明                        |
| -------------------------- | --------------------- | -------------------------------- | ---------------------- | --------------------------- |
| 最大重试次数               | 数字（int）           | 模型首次调用失败后，最多重试次数 | 2（通常总尝试=1+重试） | 控制模型调用重试上限        |
| 重试（retry_on）           | 异常类型元组 / 可调用 | 哪些异常触发重试                 | `(Exception,)`         | 可传函数：`fn(exc) -> bool` |
| 失败（on_failure）         | 字符串 / 可调用       | 所有重试失败后的处理方式         | `"continue"`           | 见下方详细说明              |
| 回退因子（backoff_factor） | 数字（float）         | 指数退避倍数                     | 2.0                    | 控制重试间隔增长速度        |
| 初始延迟（initial_delay）  | 数字（float）         | 第一次重试前等待时间（秒）       | 1.0                    | 基础延迟                    |
| 最大延迟（max_delay）      | 数字（float）         | 重试间隔最大值（秒）             | 60.0                   | 防止等待时间过长            |
| 抖动（jitter）             | 布尔（bool）          | 是否增加随机波动（±25%）         | True                   | 避免“惊群效应”              |

------

##### ⚙️ 失败处理（on_failure）详解

| 值                   | 行为       | 说明                                                    |
| -------------------- | ---------- | ------------------------------------------------------- |
| `"continue"`（默认） | 优雅降级   | 返回一个 `AIMessage`（包含错误信息），让 Agent 继续运行 |
| `"error"`            | 强制中断   | 抛出异常，直接停止 Agent                                |
| 自定义函数           | 自定义处理 | `fn(exception) -> str`，返回 AIMessage 内容             |

------

##### 🧠 指数退避机制（核心）

👉 每次重试等待时间：

```
delay = initial_delay * (backoff_factor ^ retry_number)
```

并且：

```
delay ≤ max_delay
```

如果开启抖动：

```
实际 delay = delay ± 25% 随机波动
```

------

##### 🔥 和“工具重试”的关键区别（你必须知道）

| 维度         | 模型重试                   | 工具重试                       |
| ------------ | -------------------------- | ------------------------------ |
| 作用对象     | LLM 调用                   | 外部工具调用                   |
| 返回对象     | AIMessage                  | ToolMessage                    |
| 默认失败行为 | continue（让模型自己兜底） | return_message（交给模型处理） |

------

##### 🚀 一句话总结（建议记住）

👉 **模型重试 = 给 LLM 调用加“容错层”，避免因为网络/限流等问题直接失败**





In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware

agent = create_agent(
    model=model,
    tools=[search_tool, database_tool],
    middleware=[
        ModelRetryMiddleware(
            max_retries=3,
            backoff_factor=2.0,
            initial_delay=1.0,
        ),
    ],
)

###  LLM tool emulator

#### 核心作用

通常情况下，当 Agent 决定调用一个工具（比如 `send_email`）时，系统会真实地去运行你的 Python 函数。但在开发、测试或演示阶段，你可能**并不想真的发邮件**，或者**某个工具还没写好**。

**`LLMToolEmulator` 的意义在于：** 它会拦截掉对真实工具的调用，转而让 LLM 去**“脑补”**（模拟）这个工具应该返回什么结果。

> 虽然 `LLMToolEmulator` 很方便，但要注意它会带来额外的 **Token 消耗**（因为它需要模型去思考如何模拟返回结果），并且模拟的结果有时会过于“理想化”，掩盖真实工具可能出现的报错。
>
> **可以试着运行一下这段代码，然后故意不给 `send_email` 写任何逻辑（或者写个空的 `pass`），你会惊讶地发现 Agent 依然能聊得风生水起，因为它一直在“自导自演”！**
>
> 这就是中间件的威力：它不改变原有的工具逻辑，却改变了数据的流动方式。

#### 解决问题

**节省成本/风险控制**：比如你写了一个 `delete_database` 的工具，测试时你肯定不希望它真删。用这个中间件，你可以观察 Agent 是否在正确的时间尝试删除，而不用担心数据丢失。

**并行开发**：前端和 AI 逻辑已经写好了，但后端的 API（比如 `search_database`）还没写完。通过模拟器，你可以先让整个 AI 流程跑通。

**提示词优化**：你可以通过模拟器的返回结果，观察 Agent 在面对不同“假数据”时的反应，从而调整你的 Prompt。

```python
# 原始流程：模型 -> 真实工具 -> 模型
# 中间件流程：模型 -> [LLMToolEmulator 拦截并模拟结果] -> 模型
```

```python
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator

agent = create_agent(
    model="gpt-4.1",
    tools=[get_weather, search_database, send_email],
    middleware=[
        LLMToolEmulator(),  # Emulate all tools
    ],
)
```

#### 参数解析

##### tools

要模拟的工具名称列表（字符串）或 BaseTool 实例。

- 如果为`None`空（默认值），则模拟所有工具。
- 如果列表为空`[]`，则不模拟任何工具。如果为包含工具名称/实例的数组，则仅模拟数组中的工具。

##### model

用于生成模拟工具响应的模型。可以是模型标识符字符串（例如 `<model_name>）或模型 BaseChatModel`实例。如果未指定，则默认为代理的模型。

```python
# 专门请一个“演员”来模拟工具
emulator_model = init_chat_model("qwen2.5:7b-instruct", model_provider="ollama")

agent = create_agent(
    model=main_model, # 主大脑：负责决策
    tools=[heavy_tool_1, heavy_tool_2],
    middleware=[
        LLMToolEmulator(model=emulator_model),  # 模拟器：负责演戏
    ],
)
```

###### 在“脑补”什么？

当中间件拦截到工具请求时，它会偷偷给模拟模型发一个类似这样的 **Hidden Prompt**：

> “现在 Agent 想要调用 `get_weather(city='SF')`。请你根据常识，编造一个符合该工具格式的、合理的返回结果。不要废话，直接给结果。”

如果你在 `tools` 列表里写了一个**完全不存在**的函数名（比如 `magic_teleport`），只要你在中间件里挂了 `LLMToolEmulator`，Agent 依然能跑通！

**因为它根本不需要去寻找真实的 Python 代码，它只需要模拟器给它一个结果。** 这种“先画饼，后实现”的开发方式，能让你在没有任何后端支持的情况下，先把复杂的 AI 对话流给设计出来。

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator
from langchain.tools import tool
from langchain.chat_models import init_chat_model

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    return f"Weather in {location}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email."""
    return "Email sent"


model = init_chat_model(model="gemma4:e4b", model_provider="ollama")
emulator_model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama"
)

# Emulate all tools (default behavior)
agent = create_agent(
    model=model,
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator()],
)

# Emulate specific tools only
agent2 = create_agent(
    model=model,
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator(tools=["get_weather"])],
)

# Use custom model for emulation
agent4 = create_agent(
    model=model,
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator(model=emulator_model)],
)

### Context editing 上下文编辑

当达到令牌限制时，通过清除较旧的工具调用输出来管理对话上下文，同时保留最近的结果。这有助于在包含大量工具调用的长时间对话中保持上下文窗口的可控性。上下文编辑在以下情况下非常有用：

> - 长时间的对话，期间调用了太多工具，超过了令牌限制。
> - 通过移除不再相关的旧工具输出来降低代币成本
> - 仅保留上下文中最新的 N 个工具结果



只清理“工具调用产生的冗余上下文”，而不是全部对话。

> **Agent 很“啰嗦”**：每一次工具调用（查天气、搜数据库）都会产生两条消息（Call 和 Response）。如果一个任务需要调用 10 次工具，历史记录里就会多出 20 条密密麻麻的原始数据。
>
> **模型会“迷失”**：模型能处理的上下文长度是有限的（Context Window）。如果历史记录里塞满了 50 轮之前的工具调用细节，模型不仅费钱，还容易被老数据干扰，导致“逻辑混乱”。
>
> **精准记忆**：通常模型只需要知道“刚才查到了什么”，而不需要知道“半小时前查了什么”。`ClearToolUsesEdit` 帮你实现了**只记近的，不记远的**。

#### 参数解析

| 参数名称           | 类型              | 含义                       | 默认值                  | 说明                                                |
| ------------------ | ----------------- | -------------------------- | ----------------------- | --------------------------------------------------- |
| edits              | list[ContextEdit] | 要应用的上下文编辑策略列表 | `[ClearToolUsesEdit()]` | 可以组合多个策略                                    |
| token_count_method | 字符串（str）     | token 计算方式             | `"approximate"`         | `'approximate'`（快速估算）或 `'model'`（精确计算） |

##### ClearToolUsesEdit 参数

| 参数名称          | 类型          | 含义                   | 默认值        | 说明                           |
| ----------------- | ------------- | ---------------------- | ------------- | ------------------------------ |
| trigger           | 数字（int）   | 触发清理的 token 阈值  | 100000        | 超过后开始清理旧工具输出       |
| clear_at_least    | 数字（int）   | 每次至少清理多少 token | 0             | 0 表示“按需清理”               |
| keep              | 数字（int）   | 保留最近多少条工具结果 | 3             | 这些不会被删除                 |
| clear_tool_inputs | 布尔（bool）  | 是否清除工具调用参数   | False         | True 会把参数替换为空对象 `{}` |
| exclude_tools     | list[str]     | 不参与清理的工具       | `()`          | 白名单工具永远保留             |
| placeholder       | 字符串（str） | 被清理内容的占位符     | `"[cleared]"` | 替换原始内容                   |



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit

model = init_chat_model(model="gemma4:e4b", model_provider="ollama")

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=100000,
                    keep=3,
                ),
            ],
        ),
    ],
)

### Shell Tools 

#### 核心作用：赋予 Agent 系统级执行力

`ShellToolMiddleware` 的主要作用是**允许 Agent 动态生成并运行 Shell 命令**。

通常情况下，Agent 只能运行你预定义的 Python 函数。但有了这个中间件，它就像变身成了一个“初级系统管理员”。它可以：

- **管理文件**：创建、移动、读取或删除 `/workspace` 目录下的文件。
- **运行环境**：执行 `pip install` 安装依赖，或者运行 `npm build`。
- **自动化操作**：执行复杂的 Linux 命令来处理数据或检查系统状态。



| 参数名称          | 类型                                  | 含义                                                         | 默认值                | 说明                                     |
| ----------------- | ------------------------------------- | ------------------------------------------------------------ | --------------------- | ---------------------------------------- |
| workspace_root    | `str | Path | None`                   | Shell 会话的工作目录<br />规定 Agent 只能在 `/workspace` 目录下活动。它没法通过 `cd ..` 跑去删你的系统内核。 | None                  | 不指定则创建临时目录（会话结束自动删除） |
| startup_commands  | `tuple[str] | list[str] | str | None` | 会话启动后执行的命令                                         | None                  | 按顺序执行（初始化环境）**生命周期起始** |
| shutdown_commands | `tuple[str] | list[str] | str | None` | 会话关闭前执行的命令                                         | None                  | 用于清理资源 **生命周期结束**            |
| execution_policy  | `BaseExecutionPolicy | None`          | 执行策略（权限/隔离/资源控制）                               | `HostExecutionPolicy` | 见下方详细说明                           |
| redaction_rules   | `list[RedactionRule] | None`          | 输出脱敏规则                                                 | None                  | 仅对输出生效（执行后处理）               |
| tool_description  | `str | None`                          | 工具描述覆盖                                                 | None                  | 用于提示模型如何使用该工具               |
| shell_command     | `Sequence[str] | str | None`          | 启动 shell 的命令                                            | `/bin/bash`           | 可替换为 zsh / sh / 自定义               |
| env               | `Mapping[str, Any] | None`            | 环境变量                                                     | None                  | 会被转成字符串注入 shell                 |

| 策略                        | 含义                              | 适用场景                              |
| --------------------------- | --------------------------------- | ------------------------------------- |
| HostExecutionPolicy         | 直接在宿主机执行（权限最大）      | 已在容器/沙箱中的安全环境             |
| DockerExecutionPolicy       | 每次运行启动独立 Docker 容器      | 强隔离（推荐生产）                    |
| CodexSandboxExecutionPolicy | 使用 Codex CLI 沙箱（更严格限制） | 高安全要求（限制 syscall / 文件系统） |

####  为什么要用中间件来实现？

* “我直接写一个 `run_cmd` 的 Python 工具不就行了？”

使用中间件的优雅之处在于：

- **自动注入**：它会自动为 Agent 注入一系列标准的 Shell 操作工具，不需要你手动一个一个写 `ls`, `cat`, `mkdir` 的函数。
- **全局拦截**：中间件可以统一管理所有命令的**超时时间**、**权限过滤**和**日志记录**。
- **环境一致性**：它确保 Agent 看到的路径永远是基于 `workspace_root` 的相对路径。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ShellToolMiddleware,
    HostExecutionPolicy,
    DockerExecutionPolicy,
    RedactionRule,
)

model = init_chat_model(model="gemma4:e4b", model_provider="ollama")

# Basic shell tool with host execution
agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        ShellToolMiddleware(
            workspace_root="/workspace",
            execution_policy=HostExecutionPolicy(),
        ),
    ],
)

# Docker isolation with startup commands
agent_docker = create_agent(
    model=model,
    tools=[],
    middleware=[
        ShellToolMiddleware(
            workspace_root="/workspace",
            startup_commands=["pip install requests", "export PYTHONPATH=/workspace"],
            execution_policy=DockerExecutionPolicy(
                image="python:3.11-slim",
                command_timeout=60.0,
            ),
        ),
    ],
)

# With output redaction (applied post execution)
agent_redacted = create_agent(
    model=model,
    tools=[],
    middleware=[
        ShellToolMiddleware(
            workspace_root="/workspace",
            redaction_rules=[
                RedactionRule(pii_type="api_key", detector=r"sk-[a-zA-Z0-9]{32}"),
            ],
        ),
    ],
)

###  File search 文件搜索

在文件系统上提供 Glob 和 Grep 搜索工具。文件搜索中间件可用于以下用途：

- 代码探索与分析
- 按名称模式查找文件
- 使用正则表达式搜索代码内容
- 需要进行文件发现的大型代码库

#### 参数解析

| 参数名称         | 类型          | 含义                                   | 默认值 | 说明                                       |
| ---------------- | ------------- | -------------------------------------- | ------ | ------------------------------------------ |
| root_path        | 字符串（str） | 搜索的根目录（所有文件操作基于此路径） | 必填   | 限定搜索范围，防止越界访问                 |
| use_ripgrep      | 布尔（bool）  | 是否使用 ripgrep 进行搜索              | True   | 若系统无 ripgrep，则自动回退到 Python 正则 |
| max_file_size_mb | 整数（int）   | 允许搜索的最大文件大小（MB）           | 10     | 超过该大小的文件会被跳过                   |

👉 **root_path 控边界，ripgrep 控性能，max_file_size 控资源**

#### ❗ 常见坑

##### 1️⃣ root_path 设太大

```
扫描整个磁盘 → 慢 + 不安全 ❌
```

------

##### 2️⃣ 关闭 ripgrep

```
性能直接下降一个量级 ❌
```

------

##### 3️⃣ max_file_size 太大

```
容易读到日志/二进制大文件 → 卡死 ❌
```

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import FilesystemFileSearchMiddleware
from langchain.messages import HumanMessage

model = init_chat_model(model="gemma4:e4b", model_provider="ollama")

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        FilesystemFileSearchMiddleware(
            root_path="/workspace",
            use_ripgrep=True,
            max_file_size_mb=10,
        ),
    ],
)

# Agent can now use glob_search and grep_search tools
result = agent.invoke({
    "messages": [HumanMessage("Find all Python files containing 'async def'")]
})

# The agent will use:
# 1. glob_search(pattern="**/*.py") to find Python files
# 2. grep_search(pattern="async def", include="*.py") to find async functions

###  Filesystem middleware 文件系统中间件

`FileSystemMiddleware` 就是给了 Agent 一个**文件管理器面板**。它更安全、更专注，只负责对文件进行“增删改查”。

------

#### 1. 它的核心用途

它的存在是为了让 Agent 具备**持久化数据**的能力。没有它，Agent 的知识只存在于对话中；有了它，Agent 就能读写磁盘上的文件。

- **数据分析**：读取一个本地的 CSV 文件并进行总结。
- **代码协作**：帮你写一段代码并直接保存为 `.py` 文件。
- **日志记录**：将对话中的重要信息提取出来，存入一个本地的文本文件。

#### 2. 代码示例

```python
from langchain.agents.middleware import FileSystemMiddleware

agent = create_agent(
    model=model,
    tools=[], 
    middleware=[
        FileSystemMiddleware(
            base_dir="/Users/webster/my_project", # 根目录
            allow_read=True,                      # 允许读取
            allow_write=True,                     # 允许写入
            allow_delete=False                    # 禁止删除（安全保护）
        ),
    ],
)
```

| **参数名**            | **作用解释**   | **备注**                                            |
| --------------------- | -------------- | --------------------------------------------------- |
| **`base_dir`**        | **工作根目录** | Agent 只能访问这个目录及其子目录（沙箱机制）。      |
| **`allow_read`**      | **读取权限**   | 是否允许 Agent 调用 `read_file`。                   |
| **`allow_write`**     | **写入权限**   | 是否允许 Agent 调用 `write_file` 或 `append_file`。 |
| **`allow_delete`**    | **删除权限**   | 为了安全，生产环境通常设为 `False`。                |
| **`encoding`**        | **文件编码**   | 默认为 `utf-8`。                                    |
| **`auto_create_dir`** | **自动建图**   | 如果写入的路径不存在，是否自动创建中间文件夹。      |

#### 3. 为什么不直接用 Shell Tool？

“既然 Shell 命令 `cat` 和 `echo` 也能读写文件，为什么要专门搞个文件系统中间件？”

1. **安全性（最小权限原则）**： `ShellToolMiddleware` 可能会被 AI 利用来运行 `curl` 攻击你的网络。而 `FileSystemMiddleware` 物理上限制了只能进行文件操作，没法运行网络命令。
2. **结构化输出**： 中间件会自动为 Agent 生成一组结构清晰的工具：`list_directory`, `read_file`, `write_file`。这些工具的参数定义非常明确，比让模型自己去拼凑 Shell 字符串要稳定得多。
3. **跨平台性**： Shell 命令在 Windows 和 Linux 下是不一样的（`ls` vs `dir`）。文件系统中间件屏蔽了底层差异，同一套逻辑在任何系统上都能跑通。

#### 4. 短期记忆和长期记忆

**混合存储后端（Composite Backend）**。

它不仅仅是把文件存到硬盘上，而是通过路由（Routing）机制，把 Agent 的文件操作映射到了不同的“记忆层”。你可以把它理解为给 Webster 准备了一个**“内存盘”**和一个**“云盘”**。

------

##### 1. 核心区别：短期 vs 长期

在这个配置中，区别主要体现在数据的**生命周期**和**可见范围**上：

| **维度**     | **短期文件系统 (StateBackend)**                          | **长期文件系统 (StoreBackend)**                       |
| ------------ | -------------------------------------------------------- | ----------------------------------------------------- |
| **底层映射** | 映射到当前线程的 **`State`**                             | 映射到全局的 **`InMemoryStore`**                      |
| **生命周期** | **单次对话**。对话结束或 Thread 重置，数据通常就消失了。 | **跨对话/持久化**。即使用户明天再来，数据依然在那里。 |
| **作用范围** | 仅限当前这次任务。类似于“草稿纸”。                       | 全局共享。类似于“百科全书”或“个人档案库”。            |
| **访问路径** | 除了 `/memories/` 以外的所有路径（默认）。               | 必须以 **`/memories/`** 开头的路径。                  |

------

##### 2. 为什么要搞“复合后端 (CompositeBackend)”？

它的意义在于**“自动化分层存储”**。通过 `routes` 参数，你为 Agent 制定了存储规则：

- **默认情况 (`StateBackend`)**：当 Agent 执行 `write_file(path="todo.txt", ...)` 时，它是写在当前对话的临时状态里。这适合存放临时的中间计算结果、解压的临时文件等。
- **特定路径 (`StoreBackend`)**：当 Agent 执行 `write_file(path="/memories/user_habit.txt", ...)` 时，中间件发现路径匹配到了 `/memories/`，于是自动把数据存入 `InMemoryStore`。

**`StateBackend`** 保证了 Agent 办事时手脚利索，不留垃圾。

**`StoreBackend`** 让 Agent 拥有了跨越时间的灵魂和记忆。

------

##### 3. 实战场景模拟

想象一下你正在教 Webster 认识你：

1. **临时任务**：你让 Webster 总结一份 PDF。它把 PDF 的每一页内容读取后写成 `page_1.txt`, `page_2.txt`。这些文件放在 **短期后端**，任务完成就丢掉了，不占用长期空间。
2. **长期记忆**：你告诉 Webster：“我喜欢吃川菜。” Webster 觉得这很重要，于是执行 `write_file("/memories/preferences.json", "{'food': 'Sichuan'}")`。
3. **结果**：第二天你再问 Webster 建议吃什么，它会去 `/memories/` 路径下 `ls` 一下，发现你的喜好，从而给出建议。

##### 4. `custom_tool_descriptions`

代码中还定义了 `custom_tool_descriptions`。这也是为了优雅地引导 Agent：

- 你可以明确告诉模型：“当你想要记住用户的长期习惯时，请务必在 `/memories/` 目录下创建文件。”
- 这种**“软引导”**结合**“硬路由”**，能让 Agent 的行为非常可控。



In [ ]:
from langchain.agents import create_agent
from deepagents.middleware import FilesystemMiddleware
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

model = init_chat_model(model="gemma4:e4b", model_provider="ollama")
agent = create_agent(
    model=model,
    store=store,
    middleware=[
        FilesystemMiddleware(
            backend=CompositeBackend(
                default=StateBackend(),
                routes={"/memories/": StoreBackend()}
            ),
            custom_tool_descriptions={
                "ls": "Use the ls tool when...",
                "read_file": "Use the read_file tool to..."
            }  # Optional: Custom descriptions for filesystem tools
        ),
    ],
)

### Subagent（LangChain v2 + DeepAgents 新范式）

SubAgentMiddleware = 任务路由 + 上下文隔离 + 能力分层

#### ❌ 老理解（不够准确）

```
必须提前定义好所有子代理
```

------

#### ✅ 新理解（更本质）

👉 **主 Agent 在运行时根据 description 做“语义路由”**

```
不是固定调用谁
而是：谁“最像能干这件事的人”
```

#### ✅ 标准执行流程（v2 思维）

```
1. 用户输入
2. 主 Agent 推理（LLM）：
   - 是否需要 subagent？
   - 选哪个 subagent？（基于 description）
3. 如果命中：
   → 触发 SubAgentMiddleware
4. 子 Agent 执行：
   - 自己的 prompt
   - 自己的 tools
5. 返回结果（结构化）
6. 主 Agent 整合 & 输出
```

> SubAgentMiddleware 本质是一个“语义路由 + 能力分层”的多 Agent 架构。
>
> 主 Agent 负责理解与调度，
> 子 Agent 负责垂直执行与工具调用，
> 通过上下文隔离 + 工具作用域隔离，实现更高的准确率与可扩展性。

In [1]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from deepagents.middleware.subagents import SubAgentMiddleware, SubAgent
from deepagents.backends import StateBackend
from langchain.agents import create_agent

# 1. 定义工具
@tool
def get_weather(city: str) -> str:
    """获取指定城市的实时天气状况。"""
    return f"{city} 的天气是：晴朗，25°C。"

# 2. 初始化模型
model = init_chat_model(model="gemma4:e4b", model_provider="ollama")

# 3. 【核心修复】定义至少一个子代理模板
# 主代理会根据 description 来决定是否把任务外包给它
weather_worker = SubAgent(
    name="weather_specialist",
    description="专门负责查询天气和气象建议的助手。",
    system_prompt="你是一个专业的气象员。请使用 get_weather 工具为用户查询信息。",
    tools=[get_weather],
    model=init_chat_model(
        model="qwen2.5:7b-instruct-q4_K_M",
        model_provider="ollama"
    )
)
# 4. 配置中间件
subagent_mgr = SubAgentMiddleware(
    backend=StateBackend(),
    subagents=[weather_worker],  # 传入刚才定义的子代理
)

# 5. 创建 Webster 主代理
agent = create_agent(
    model=model,
    tools=[], 
    middleware=[subagent_mgr],
)

if __name__ == "__main__":
    inputs = {"messages": [{"role": "user", "content": "帮我查一下上海的天气。"}]}
    
    from langchain_core.messages import AIMessage, ToolMessage

    for chunk in agent.stream(inputs, stream_mode="updates", version="v2"):
        data = chunk.get("data", {})

        # 模型输出
        if "model" in data:
            for msg in data["model"]["messages"]:
                if isinstance(msg, AIMessage):
                    if msg.tool_calls:
                        print(f"\n🤖 模型决策调用工具：{msg.tool_calls}\n")
                    elif msg.content:
                        print(f"\n🤖 最终回答：{msg.content}\n")

        # 工具输出（包括子代理）
        if "tools" in data:
            for msg in data["tools"]["messages"]:
                if isinstance(msg, ToolMessage):
                    print(f"\n🛠️ 工具结果：{msg.content}\n")

e:\code\LangChainDemo\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
e:\code\LangChainDemo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.



🤖 模型决策调用工具：[{'name': 'task', 'args': {'description': '请查询上海目前的实时天气状况，包括温度、天气描述和未来几天的天气预报。请以清晰的格式返回所有信息。', 'subagent_type': 'weather_specialist'}, 'id': '2e11ab3d-5b8b-4ef1-81f0-22eaeb7b899a', 'type': 'tool_call'}]


🛠️ 工具结果：目前上海市的实时天气状况如下：
- 天气描述：晴朗
- 温度：25°C

由于当前仅提供了即时的天气情况，未来几天的天气预报信息未能直接获取。您需要我进一步查询吗？或者我可以帮您提供一些查看长期天气预报的方法或网站建议。


🤖 最终回答：根据我查询的结果，上海目前的实时天气情况是：

*   **天气描述：** 晴朗
*   **温度：** 25°C

请注意，当前查询结果只包含了即时的天气状况，未能获取未来几天的完整预报信息。

您是否需要我帮您查找查看未来几天天气预报的可靠网站资源，或者您还有其他想了解的天气相关信息吗？



In [2]:
inputs = {"messages": [{"role": "user", "content": "帮我查一下上海的天气。"}]} # 运行流式输出 
for chunk in agent.stream(inputs, stream_mode="updates", version="v2"): 
    print(chunk)

{'type': 'updates', 'ns': (), 'data': {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-12T05:56:50.949799Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13018240500, 'load_duration': 307494600, 'prompt_eval_count': 2117, 'prompt_eval_duration': 76492600, 'eval_count': 239, 'eval_duration': 12390111400, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d8043-75a9-7be3-a1c5-b3808374ca58-0', tool_calls=[{'name': 'task', 'args': {'description': '请查询上海当前的天气情况、天气预报（例如未来3天），并提供一些相关的穿着建议和生活建议。请用清晰、易懂的格式返回最终的完整报告。', 'subagent_type': 'weather_specialist'}, 'id': '955bc2af-335e-420b-9c56-1d6e97878fc8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2117, 'output_tokens': 239, 'total_tokens': 2356})]}}}
{'type': 'updates', 'ns': (), 'data': {'tools': {'messages': [ToolMessage(content='上海当前的天气为晴朗，气温约为25°C。\n\n接下来我将继续查询未来三天的详细天气预报，并为您提供